In [0]:
#Step 1: Environment Setup & Data Ingestion
'''
Step 1: Environment Setup & Data Ingestion
Install GeoPandas, import all required spatial libraries, load the Paris datasets (quartier_paris.geojson, velib-disponibilite-en-temps-reel.geojson, voie.geojson, and construction/chantiers) into GeoDataFrames, verify Coordinate Reference Systems (CRS), and inspect the data structure.   

Step 2: Basic Spatial Operations & Filtering
filter features by attributes, calculate geometry metrics (lengths of streets, surface areas of quartiers), and run basic aggregations.  

Step 3: Spatial Relationships & Spatial Joins (sjoin)
Perform point-in-polygon queries, intersect stations with quartiers, count the exact number of Vélib stations per quartier/arrondissement, and identify roads or active construction zones affecting bike docks.   

Step 4: Advanced Aggregations & Availability Metrics
Compute capacity, real-time bike availability, and electric vs. mechanical ratios per quartier; aggregate spatial statistics across neighborhoods; and identify the most utilized or saturated stations.   

Step 5: Spatial Data Visualization & Map Creation
Plot layered maps of Paris showing quartier boundaries, road networks, stations, and construction zones; build choropleth maps (e.g., gradient coloring of quartiers by station density or bike availability); and generate styled publication-ready visual exports.   '''

# Step 1: Environment Setup & Data Ingestion

In [0]:
# installing libraries in Databricks
# %pip install geopandas
# %pip install geopandas matplotlib contextily
# %pip install folium

In [0]:
# import libraries
import pandas as pd
import geopandas as gpd
import numpy as np 
import matplotlib.pyplot as plt
import contextily as cx
import folium

In [0]:
# Load GeoJSON files into GeoDataFrames

gdf_quartiers = gpd.read_file('/Volumes/workspace/geojson_files/geojsons/quartier_paris.geojson')

gdf_velib = gpd.read_file('/Volumes/workspace/geojson_files/geojsons/velib-emplacement-des-stations.geojson')

gdf_fermeture = gpd.read_file('/Volumes/workspace/geojson_files/geojsons/fermetures-voirie.geojson')

gdf_chantiers = gpd.read_file('/Volumes/workspace/geojson_files/geojsons/chantiers-perturbants.geojson')

gdf_arrondissements = gpd.read_file('/Volumes/workspace/geojson_files/geojsons/arrondissements.geojson')




In [0]:
# inspect gdf_quartiers 
gdf_quartiers.head(5)

In [0]:
# inspect gdf velib 
gdf_velib.head(5)

In [0]:
# inspect gdf_arrondissements 
gdf_arrondissements.head(5)

In [0]:
# inspect gdf_chantiers
gdf_chantiers.head(5)

In [0]:
# inspect gdf_fermeture
gdf_fermeture.head(5)

In [0]:
# check the data type for arrondissemnts
gdf_arrondissements.dtypes

In [0]:
# check data type for velib 
gdf_velib.dtypes


In [0]:
# check the CRS of the geodataframes 
print(gdf_arrondissements.crs)
print(gdf_chantiers.crs)
print(gdf_fermeture.crs)
print(gdf_quartiers.crs)
print(gdf_velib.crs)

# all the GeoDataFrames have the same CRS

In [0]:
# geometry types in qurtiers
gdf_quartiers.geom_type.value_counts()


In [0]:
# geometry types  in velib
gdf_velib.geom_type.value_counts()

In [0]:
# Create a base Folium map centered on Paris
paris_map = folium.Map(
    location=[48.8566, 2.3522],
    zoom_start=12,
    tiles="OpenStreetMap"
)

# show the map 
paris_map

In [0]:
#  Count missing geometries (Null / None)
print(gdf_chantiers.geometry.isna().sum())

In [0]:
# Count empty geometries (e.g., POINT EMPTY)
print(gdf_chantiers.geometry.is_empty.sum())

In [0]:
#  Count duplicate station codes in Velib
gdf_velib["stationcode"].duplicated().sum()

# Step 2:  Spatial Operations & Filtering


In [0]:
# filter gdf_quartiers to select only the quartiers located in the 1st arrondissement
gdf_quartiers[gdf_quartiers['c_ar']==1]

In [0]:
# filter gdf_quartiers to select quartiers in MULTIPLE arrondissements at once (3,4,8)
gdf_quartiers[gdf_quartiers['c_ar'].isin([3,4,8])]

In [0]:
# transform the CRS of the geodataframes

gdf_quartiers_m = gdf_quartiers.to_crs(epsg=2154)
gdf_arrondissements_m = gdf_arrondissements.to_crs(epsg=2154)
gdf_chantiers_m = gdf_chantiers.to_crs(epsg=2154)
gdf_fermeture_m = gdf_fermeture.to_crs(epsg=2154)
gdf_velib_m = gdf_velib.to_crs(epsg=2154)
print(gdf_chantiers_m.crs)

In [0]:
# calculate the surface area of each quartier in square meters
gdf_quartiers_m["area_m2"] = gdf_quartiers_m.geometry.area
gdf_quartiers_m.head()

In [0]:
# calculate the outer perimeter of each quartier in meters
gdf_quartiers_m["perimeter_m"] = gdf_quartiers_m.geometry.length
gdf_quartiers_m.head(5)

In [0]:
# calculate the total land area of the entire city
gdf_quartiers_m["area_m2"].sum()

In [0]:
# Visualize quartiers in paris map 
import copy

map_quartiers = copy.deepcopy(paris_map)

folium.GeoJson(gdf_quartiers).add_to(map_quartiers)

# 3. Display the map
map_quartiers



In [0]:
# Visualize arrondissements in paris 

map_arrondissements = copy.deepcopy(paris_map)

folium.GeoJson(gdf_arrondissements).add_to(map_arrondissements)

# 3. Display the map
map_arrondissements

In [0]:
# Visualize velib stations in paris 

map_velib = copy.deepcopy(paris_map)

folium.GeoJson(
    gdf_velib,
    marker=folium.CircleMarker(radius=3, fill=True)
).add_to(map_velib)

# 3. Display the map
map_velib

In [0]:
# Visualize  chantiers in paris 

map_chantiers = copy.deepcopy(paris_map)

folium.GeoJson(
    gdf_chantiers.dropna(subset=["geometry"]).geometry
).add_to(map_chantiers)

# 3. Display the map
map_chantiers

# Step 3: Spatial Relationships & Spatial Joins